In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install transformers torch scikit-learn tqdm

In [8]:
from transformers import AutoTokenizer, AutoModel
import torch
import os
import json
import numpy as np
from tqdm import tqdm
from torch.nn.functional import cosine_similarity

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

def encode(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state[:, 0, :]  # CLS

def load_cases(folder_path):
    cases = {}
    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                cases[file] = f.read()
    return cases

def precompute_embeddings(cases):
    embeddings = {}
    for cid, text in tqdm(cases.items(), desc="Encoding cases"):
        embeddings[cid] = encode(text)
    return embeddings

def fast_rank(query_id, embeddings):
    query_vec = embeddings[query_id]

    scores = []
    for cid, vec in embeddings.items():
        if cid == query_id:
            continue

        score = cosine_similarity(query_vec, vec).item()
        scores.append((cid, score))

    ranked = sorted(scores, key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in ranked]

def _to_float(x):
    try:
        return float(x)
    except Exception:
        return x


def evaluate_with_yf(
    cases,
    ground_truth,
    embeddings,
    candidate_dict=None,
    topk=5,
    log_path=None,
    method_name="LEGAL-BERT"
):
    correct_pred = retri_cases = relevant_cases = 0
    macro_p = macro_r = macro_f = 0

    correct_pred_yf = retri_cases_yf = relevant_cases_yf = 0
    macro_p_yf = macro_r_yf = macro_f_yf = 0

    num_queries = 0
    missing_queries = []
    missing_docs = set()

    for query_id, relevant_list in ground_truth.items():

        if query_id not in cases:
            missing_queries.append(query_id)
            continue

        for rel in relevant_list:
            if rel not in cases:
                missing_docs.add(rel)

        # -------- FULL RANKING (NO FILTER) --------
        ranked_full = fast_rank(query_id, embeddings)
        top_k = ranked_full[:topk]

        relevant_set = set(relevant_list)

        # -------- NORMAL METRICS --------
        tp = len(set(top_k) & relevant_set)

        correct_pred += tp
        retri_cases += topk
        relevant_cases += len(relevant_set)

        p = tp / topk
        r = tp / len(relevant_set) if relevant_set else 0
        f = 2*p*r/(p+r) if (p+r)>0 else 0

        macro_p += p
        macro_r += r
        macro_f += f

        # -------- YEAR FILTER (ONLY HERE) --------
        if candidate_dict:
            cand_set = set(candidate_dict[query_id])

            ranked_yf = [c for c in ranked_full if c in cand_set]
            top_k_yf = ranked_yf[:topk]

            tp_yf = len(set(top_k_yf) & relevant_set)

            correct_pred_yf += tp_yf
            retri_cases_yf += topk
            relevant_cases_yf += len(relevant_set)

            p_yf = tp_yf / topk
            r_yf = tp_yf / len(relevant_set) if relevant_set else 0
            f_yf = 2*p_yf*r_yf/(p_yf+r_yf) if (p_yf+r_yf)>0 else 0

            macro_p_yf += p_yf
            macro_r_yf += r_yf
            macro_f_yf += f_yf

        num_queries += 1

    # -------- FINAL METRICS --------
    micro_pre = correct_pred / retri_cases if retri_cases else 0
    micro_recall = correct_pred / relevant_cases if relevant_cases else 0
    micro_f = 2*micro_pre*micro_recall/(micro_pre+micro_recall) if (micro_pre+micro_recall)>0 else 0

    macro_pre = macro_p / num_queries if num_queries else 0
    macro_recall = macro_r / num_queries if num_queries else 0
    macro_f = macro_f / num_queries if num_queries else 0

    if candidate_dict:
        micro_pre_yf = correct_pred_yf / retri_cases_yf if retri_cases_yf else 0
        micro_recall_yf = correct_pred_yf / relevant_cases_yf if relevant_cases_yf else 0
        micro_f_yf = 2*micro_pre_yf*micro_recall_yf/(micro_pre_yf+micro_recall_yf) if (micro_pre_yf+micro_recall_yf)>0 else 0

        macro_pre_yf = macro_p_yf / num_queries if num_queries else 0
        macro_recall_yf = macro_r_yf / num_queries if num_queries else 0
        macro_f_yf = macro_f_yf / num_queries if num_queries else 0
    else:
        micro_pre_yf = micro_recall_yf = micro_f_yf = 0
        macro_pre_yf = macro_recall_yf = macro_f_yf = 0

    # -------- LOGGING --------
    log_lines = [
        f"Method: {method_name}",
        f"TopK: {topk}",
        f"Queries: {num_queries}",
    ]

    if missing_docs:
        log_lines.append(f"Missing docs in pool: {len(missing_docs)}")
    if missing_queries:
        log_lines.append(f"Missing query files: {len(missing_queries)}")

    log_lines += [
        f"Correct Predictions:  {correct_pred}",
        f"Retrieved Cases:  {retri_cases}",
        f"Relevant Cases:  {relevant_cases}",
        f"Micro Precision:  {micro_pre}",
        f"Micro Recall:  {micro_recall}",
        f"Micro F1:  {micro_f}",
        f"Macro Precision:  {macro_pre}",
        f"Macro Recall:  {macro_recall}",
        f"Macro F1:  {macro_f}",
    ]

    if candidate_dict:
        log_lines += [
            f"Correct Predictions yf:  {correct_pred_yf}",
            f"Retrieved Cases yf:  {retri_cases_yf}",
            f"Relevant Cases yf:  {relevant_cases_yf}",
            f"Micro Precision yf:  {micro_pre_yf}",
            f"Micro Recall yf:  {micro_recall_yf}",
            f"Micro F1 yf:  {micro_f_yf}",
            f"Macro Precision yf:  {macro_pre_yf}",
            f"Macro Recall yf:  {macro_recall_yf}",
            f"Macro F1 yf:  {macro_f_yf}",
        ]

    log_lines.append("")

    print("\n".join(log_lines))

    if log_path:
        with open(log_path, "a", encoding="utf-8") as f:
            f.write("\n".join(log_lines))
        print(f"Log saved to: {log_path}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Load data
cases = load_cases("/content/drive/MyDrive/CaseGNNDataset/task1_test_2017/summary_test_2017_txt/")
labels = json.load(open("/content/drive/MyDrive/CaseGNNDataset/task1_test_2017/task1_test_labels_2017.json"))
candidate_dict = json.load(open("/content/drive/MyDrive/CaseGNNDataset/task1_test_2017/test_2017_candidate_with_yearfilter.json"))

In [10]:
# Precompute embeddings
embeddings = precompute_embeddings(cases)

Encoding cases: 100%|██████████| 168/168 [00:03<00:00, 47.67it/s]


In [11]:
evaluate_with_yf(
    cases=cases,
    ground_truth=labels,
    embeddings=embeddings,
    candidate_dict=candidate_dict,
    topk=5,
    log_path="/content/legalbert.log"
)

Method: LEGAL-BERT
TopK: 5
Queries: 38
Correct Predictions:  13
Retrieved Cases:  190
Relevant Cases:  132
Micro Precision:  0.06842105263157895
Micro Recall:  0.09848484848484848
Micro F1:  0.08074534161490683
Macro Precision:  0.06842105263157895
Macro Recall:  0.09605263157894736
Macro F1:  0.07485380116959064
Correct Predictions yf:  17
Retrieved Cases yf:  190
Relevant Cases yf:  132
Micro Precision yf:  0.08947368421052632
Micro Recall yf:  0.12878787878787878
Micro F1 yf:  0.10559006211180125
Macro Precision yf:  0.08947368421052632
Macro Recall yf:  0.1219298245614035
Macro F1 yf:  0.09780701754385966

Log saved to: /content/legalbert.log
